# 04b. Causal attention как nn.Module

Лекция: [Свой модуль PyTorch](../../notes/interview-prep/08c-attention-module.md).

Causal mask уже работает. Теперь переносим те же вычисления в переиспользуемый слой. Notebook независим от предыдущего. Сегодня выполни **только остановку A**. B/C оставлены для следующих шагов разбора.

Здесь одна голова, d_k=d_v=d_model, без bias, dropout, padding и выходной проекции. На вход подаём готовые векторы X; embeddings не входят в этот модуль.

## A1. Минимальный пример: слой живёт внутри объекта

`__init__` создаёт параметры один раз при создании экземпляра. `forward` описывает вычисление на очередном входе. Присваивание nn.Linear атрибуту self регистрирует его как дочерний модуль.

Вызываем `model(x)`, а не `model.forward(x)`: стандартный вызов модуля включает механизм hooks. Сам по себе вызов не обучает модель.

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(42)

class TokenProjection(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.projection = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x):
        return self.projection(x)

example = TokenProjection(4, 2)
x_example = torch.randn(2, 3, 4)
y_example = example(x_example)
print(example)
print('Output shape:', y_example.shape)
for name, parameter in example.named_parameters():
    print(name, tuple(parameter.shape))
assert y_example.shape == (2, 3, 2)

TokenProjection(
  (projection): Linear(in_features=4, out_features=2, bias=False)
)
Output shape: torch.Size([2, 3, 2])
projection.weight (2, 4)


## A2. Создай контейнер трёх проекций

Заполни только три строки в __init__. Используй `nn.Linear(d_model, d_model, bias=False)` для каждой проекции. Слои должны быть разными экземплярами.

В этой остановке forward намеренно не реализован: создание экземпляра и просмотр параметров работают без forward. Полный класс напишем отдельно в B.

In [2]:
class AttentionParameters(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)

container = AttentionParameters(d_model=4)
for attr in ['query_projection', 'key_projection', 'value_projection']:
    layer = getattr(container, attr)
    assert isinstance(layer, nn.Linear), f'Заполни {attr}'
    assert layer.in_features == layer.out_features == 4
    assert layer.bias is None

assert container.query_projection.weight is not container.key_projection.weight
assert container.query_projection.weight is not container.value_projection.weight
assert container.key_projection.weight is not container.value_projection.weight
parameter_count = sum(p.numel() for p in container.parameters())
assert parameter_count == 48
print('Параметров:', parameter_count)
print('state_dict keys:', list(container.state_dict()))

Параметров: 48
state_dict keys: ['query_projection.weight', 'key_projection.weight', 'value_projection.weight']


### Остановка A — сохрани notebook и пришли на проверку

1. Зачем вызывать super().__init__() перед присваиванием дочерних слоёв?
2. Почему создаём nn.Linear в __init__, а не заново при каждом forward?
3. Откуда container.parameters() узнаёт о трёх матрицах?
4. Почему здесь 48 параметров, независимо от B и T?

**Мои ответы:**

1. Он инициализирует модуль. Это вызов конструктора базового класса. Нужно вызывать в начале, чтоюы потом корректно сохранились параметры модулей, которые инициализируем после
2. Слои создаём один раз, чтобы повторно использовать и обучать одни и те же параметры. В forward применяем существующие слои к входу. Создание нового nn.Linear при каждом forward создаёт новые случайные параметры вместо использования уже обучаемых.
3. Присваивание nn.Linear атрибуту self регистрирует дочерний модуль, внутри которого weight уже является зарегистрированным nn.Parameter. parameters() рекурсивно обходит зарегистрированные модули и возвращает существующие параметры, а не вычисляет их заново и не анализирует код конструктора.
4. Потому что у нас 3 матрицы 4\*4. Мы храним тензоры d_model\*d_model

Не нужно пока выполнять B/C. Параметры созданы, но attention этот контейнер ещё не вычисляет.

## B. Полный модуль causal attention

Перенеси три слоя в __init__, а операции из attend прошлого notebook — в forward. В forward не обращайся к глобальным X, T и слоям.

Маску строим заново из длины текущего x на x.device. Пока это проще, чем заранее хранить буфер максимального размера. Возвращаем только context.

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model:
            raise ValueError('Ожидается x формы (B,T,d_model)')
        if x.shape[1] == 0:
            raise ValueError('Последовательность должна быть непустой')

        q = self.query_projection(x)
        k = self.key_projection(x)
        v = self.value_projection(x)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked_mask = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked_mask, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        return context

In [4]:
# Проверяем формы и совпадение с явным эталоном на тех же параметрах.
model = CausalSelfAttention(d_model=4)
x = torch.randn(2, 3, 4)
output = model(x)
assert isinstance(output, torch.Tensor)
assert output.shape == x.shape
assert sum(p.numel() for p in model.parameters()) == 48

# Эталон использует те же веса, но другой способ построения маски.
q_ref = model.query_projection(x)
k_ref = model.key_projection(x)
v_ref = model.value_projection(x)
scores_ref = (q_ref @ k_ref.transpose(-2, -1)) / math.sqrt(4)
mask_ref = torch.ones(3, 3, dtype=torch.bool, device=x.device).triu(diagonal=1)
expected = torch.softmax(scores_ref.masked_fill(mask_ref, float('-inf')), dim=-1) @ v_ref
torch.testing.assert_close(output, expected)

for batch_size, length in [(1, 1), (3, 5)]:
    test_x = torch.randn(batch_size, length, 4)
    test_output = model(test_x)
    assert test_output.shape == test_x.shape
    torch.testing.assert_close(test_output[:, 0], model.value_projection(test_x)[:, 0])
print('Совпадение с эталоном и разные B/T: OK')

Совпадение с эталоном и разные B/T: OK


### Остановка B

1. Чем model.query_projection.weight отличается от q внутри forward?
2. Почему изменение длины x не требует создавать новый model?
3. Почему embeddings не входят в этот слой?

**Мои ответы:**

1. model.query_projection.weight — обучаемая матрица (D,D), общая для всех позиций. q — результат применения проекции к текущему входу, форма (B,T,D). q[b,t] — вектор одной позиции; весь q пересчитывается при каждом forward.
2. Параметры проекций зависят от d_model, а не от B или T. Одна матрица применяется к каждому входному вектору. Маска строится из текущего x.shape[1], поэтому её размер также адаптируется. Последняя размерность x должна совпадать с d_model. Broadcasting участвует в вычислении, но основная причина — независимость формы параметров от длины последовательности.
3. Это разделение ответственности: embeddings превращают token IDs в векторы, а attention принимает готовые векторы. В полной модели последующие блоки получают выходы предыдущих блоков, а не снова token IDs. Поэтому embeddings вынесены во внешнюю модель; это выбор архитектуры, не ограничение PyTorch.

## C. Причинность, градиенты и перенос параметров

Готовые тесты. Здесь искусственный loss снова проверяет механику обучения, а не качество языка. Не требуется снижение loss как доказательство корректности attention.

In [5]:
with torch.no_grad():
    changed_x = x.clone()
    changed_x[:, -1] += torch.tensor([10., -7., 5., 3.], device=x.device)
    torch.testing.assert_close(model(x)[:, :-1], model(changed_x)[:, :-1])
print('Причинность: OK')

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad(set_to_none=True)
probe_loss = model(x).square().mean()
probe_loss.backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None
    assert torch.isfinite(parameter.grad).all()
    print(name, 'grad norm:', parameter.grad.norm().item())
before_step = {name: p.detach().clone() for name, p in model.named_parameters()}
optimizer.step()
assert any(not torch.equal(before_step[name], p.detach()) for name, p in model.named_parameters())
print('Шаг оптимизатора изменил параметры: OK')

# Копирование состояния в другой экземпляр, без записи файла.
replica = CausalSelfAttention(d_model=4)
replica.load_state_dict(model.state_dict())
with torch.no_grad():
    torch.testing.assert_close(replica(x), model(x))
print('Перенос параметров через state_dict: OK')

Причинность: OK
query_projection.weight grad norm: 0.014873956330120564
key_projection.weight grad norm: 0.015358314849436283
value_projection.weight grad norm: 0.31177693605422974
Шаг оптимизатора изменил параметры: OK
Перенос параметров через state_dict: OK


### Остановка C

1. Что делает backward, а что делает step?
2. Содержит ли state_dict промежуточные q/k/v текущего forward?
3. Почему одинаковые параметры дали одинаковый выход у двух экземпляров здесь? Что могло бы изменить это при добавлении dropout?

**Мои ответы:**

1. probe_loss.backward() вычисляет производные текущего скалярного loss по параметрам через граф forward и накапливает их в parameter.grad у обучаемых листовых параметров. Значения параметров не меняются. optimizer.step() использует эти градиенты и правило оптимизатора для обновления переданных ему параметров. Здесь обычный SGD: parameter_new = parameter_old - 0.01 * parameter.grad. step() сам не вызывает backward и не очищает градиенты.

Перед новым независимым шагом optimizer.zero_grad(set_to_none=True) очищает старые градиенты, устанавливая .grad в None, но не обнуляет веса. Без очистки backward прибавляет новые градиенты к старым. В нашем тесте loss = model(x).square().mean() искусственный: проверяет вычисление градиентов и обновление параметров, но не учит предсказывать следующий токен.

2. Нет. state_dict содержит параметры и persistent buffers, а локальные q/k/v — промежуточные результаты текущего forward. Они вычисляются заново для входа. Здесь state_dict содержит три матрицы проекций. Состояние оптимизатора в model.state_dict() тоже не входит.

3. После load_state_dict два экземпляра имеют одинаковые параметры и выполняют одинаковые операции на одном входе. В нашем примере без dropout их выходы совпадают в пределах допуска проверки. Dropout в train-режиме может использовать разные случайные маски и дать разные выходы даже с одинаковыми весами. Для такого сравнения с dropout нужно перевести оба экземпляра в eval(); torch.no_grad() сам по себе dropout не отключает.

Ответы уточнены с ментором. Заполненный текст не подтверждает выполнение тестов части C: их нужно отдельно запустить и сохранить результаты.

После заполнения всех частей: Restart Kernel → Run All → сохранить outputs. Следующий этап — multi-head attention. Этот модуль ещё не полный Transformer-блок.